# Configure a Simulation

A FraudTwin simulation is controlled by a YAML configuration. Start with the small example configuration, change a few values, and generate a new deterministic run.

## 1. Choose a configuration

FraudTwin includes a small default configuration inside the installed package. Use it directly here; custom YAML files can be passed to `generate()` when needed.

In [1]:
import fraudtwin

print(fraudtwin.__version__)
print("Using the bundled minimal configuration")

0.19.0
Using the bundled minimal configuration


## 2. Generate the configured world

Generation stays in memory by default, so this does not create files.

In [2]:
data = fraudtwin.generate("../../configs/minimal.yaml")
print(data.run_id)

RUN-19652188a1efbe6c


## 3. See what the configuration produced

The manifest records the seed, configuration hash, and generated counts. For a custom file, use `fraudtwin.generate("my-config.yaml")`.

In [3]:
print("Seed:", data.manifest.seed)
print("Configuration hash:", data.manifest.scenario_config_hash)
print("Customers:", data.manifest.entity_counts["customers"])
print("Payments:", data.manifest.event_counts["payments"])

Seed: 42
Configuration hash: 19652188a1efbe6ccc5b9862e34099fb9fa0f727e83c623b02f0ceddbf75300b
Customers: 10
Payments: 100


## 4. Compare payment rails

The same configured world can contain card and PIX-like payments. Each rail has its own lifecycle events, which are useful when building event-based fraud pipelines.

Card, PIX, and account transfers are separate payment rails. This table shows how many payments were generated for each rail.

In [4]:
import polars as pl

payments_by_rail = (
    pl.DataFrame({"Payment rail": [p.payment_rail for p in data.behavior.payments]})
    .group_by("Payment rail")
    .len()
    .rename({"len": "Payments"})
    .sort("Payments", descending=True)
)

payments_by_rail

Payment rail,Payments
str,u32
"""CARD""",63
"""ACCOUNT_TRANSFER""",25
"""PIX""",12


Each payment can produce several events as it moves through its rail. These are the eight most common lifecycle events in this run.

In [5]:
events = [event.event_type for event in data.behavior.payment_events]
event_counts = (
    pl.DataFrame(
        {
            "Lifecycle event": events,
        }
    )
    .group_by("Lifecycle event")
    .len()
    .rename({"len": "Events"})
    .sort("Events", descending=True)
    .head(8)
)

event_counts

Lifecycle event,Events
str,u32
"""CARD_AUTHORIZATION_REQUESTED""",63
"""CARD_PAYMENT_INITIATED""",63
"""CARD_AUTHORIZED""",50
"""CARD_CLEARED""",47
"""CARD_CAPTURED""",47
"""CARD_SETTLED""",47
"""TRANSFER_COMPLETED""",25
"""CARD_DECLINED""",13


## 5. Reproducibility

The same configuration and seed produce the same run ID. Changing the seed creates a new reproducible run.

In [6]:
same_run = fraudtwin.generate()
print(data.run_id == same_run.run_id)

True
